# Data Vortex — Phase 2: SQL Challenge 7
## Audience Reach vs. Engagement

### 1. Challenge Description
Analyze whether creators with larger follower counts demonstrate higher average post engagement (likes, shares, comments).

Audience Groups:
- **Small Audience:** < 10,000 followers
- **Medium Audience:** 10,000–29,999 followers
- **Large Audience:** 30,000+ followers

Also evaluates manual Pearson correlation in SQLite between `follower_count` and interaction metrics.

In [ ]:
import os
import sqlite3
import pandas as pd

# File Paths
BASE_DIR = os.path.abspath(os.path.join(os.getcwd(), ".."))
DB_PATH = os.path.join(BASE_DIR, "data", "data_vortex.db")
SQL_PATH = os.path.join(BASE_DIR, "sql", "challenge_07_audience_reach_vs_engagement.sql")

print(f"Target Database: {DB_PATH}")
print(f"SQL Script:      {SQL_PATH}")

# Connect to SQLite
conn = sqlite3.connect(DB_PATH)
conn.execute("PRAGMA foreign_keys = ON;")
print("Connected to SQLite database successfully.")

### 2. Audience Group Summary & Interaction Metrics
Aggregates creator counts, percentage shares, average follower counts, and interaction averages per post across audience tiers.

In [ ]:
q_summary = """
WITH creator_groups AS (
    SELECT 
        user_id,
        follower_count,
        CASE
            WHEN follower_count < 10000 THEN 'Small Audience'
            WHEN follower_count BETWEEN 10000 AND 29999 THEN 'Medium Audience'
            WHEN follower_count >= 30000 THEN 'Large Audience'
        END AS audience_group
    FROM users
),
group_creators AS (
    SELECT 
        audience_group,
        COUNT(*) AS creator_count,
        ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM users), 2) AS percentage_of_creators,
        ROUND(AVG(follower_count), 2) AS avg_follower_count
    FROM creator_groups
    GROUP BY audience_group
),
group_posts AS (
    SELECT 
        cg.audience_group,
        COUNT(p.post_id) AS total_posts,
        ROUND(AVG(p.likes), 2) AS avg_likes_per_post,
        ROUND(AVG(p.shares), 2) AS avg_shares_per_post,
        ROUND(AVG(p.comments), 2) AS avg_comments_per_post
    FROM posts p
    INNER JOIN creator_groups cg ON p.user_id = cg.user_id
    GROUP BY cg.audience_group
)
SELECT 
    gc.audience_group,
    gc.creator_count,
    gc.percentage_of_creators,
    gc.avg_follower_count,
    gp.total_posts,
    gp.avg_likes_per_post,
    gp.avg_shares_per_post,
    gp.avg_comments_per_post
FROM group_creators gc
INNER JOIN group_posts gp ON gc.audience_group = gp.audience_group
ORDER BY 
    CASE gc.audience_group
        WHEN 'Small Audience' THEN 1
        WHEN 'Medium Audience' THEN 2
        WHEN 'Large Audience' THEN 3
    END;
"""

df_summary = pd.read_sql_query(q_summary, conn)
df_summary

### 3. Pearson Correlation Results (Manual SQLite Aggregation)
Computes Pearson correlation coefficient between follower count and engagement metrics using standard covariance and standard deviation expansion in SQL.

In [ ]:
q_corr = """
WITH creator_stats AS (
    SELECT 
        u.user_id,
        CAST(u.follower_count AS REAL) AS followers,
        AVG(p.likes) AS avg_likes,
        AVG(p.shares) AS avg_shares,
        AVG(p.comments) AS avg_comments
    FROM users u
    INNER JOIN posts p ON u.user_id = p.user_id
    GROUP BY u.user_id, u.follower_count
),
corr_likes AS (
    SELECT 
        COUNT(*) AS n_likes,
        ROUND(
            (COUNT(*) * SUM(followers * avg_likes) - SUM(followers) * SUM(avg_likes)) /
            (SQRT(COUNT(*) * SUM(followers * followers) - SUM(followers) * SUM(followers)) *
             SQRT(COUNT(*) * SUM(avg_likes * avg_likes) - SUM(avg_likes) * SUM(avg_likes))),
            4
        ) AS pearson_r_likes
    FROM creator_stats
    WHERE avg_likes IS NOT NULL
),
corr_shares AS (
    SELECT 
        COUNT(*) AS n_shares,
        ROUND(
            (COUNT(*) * SUM(followers * avg_shares) - SUM(followers) * SUM(avg_shares)) /
            (SQRT(COUNT(*) * SUM(followers * followers) - SUM(followers) * SUM(followers)) *
             SQRT(COUNT(*) * SUM(avg_shares * avg_shares) - SUM(avg_shares) * SUM(avg_shares))),
            4
        ) AS pearson_r_shares
    FROM creator_stats
    WHERE avg_shares IS NOT NULL
),
corr_comments AS (
    SELECT 
        COUNT(*) AS n_comments,
        ROUND(
            (COUNT(*) * SUM(followers * avg_comments) - SUM(followers) * SUM(avg_comments)) /
            (SQRT(COUNT(*) * SUM(followers * followers) - SUM(followers) * SUM(followers)) *
             SQRT(COUNT(*) * SUM(avg_comments * avg_comments) - SUM(avg_comments) * SUM(avg_comments))),
            4
        ) AS pearson_r_comments
    FROM creator_stats
    WHERE avg_comments IS NOT NULL
)
SELECT 
    cl.n_likes,
    cl.pearson_r_likes,
    cs.n_shares,
    cs.pearson_r_shares,
    cc.n_comments,
    cc.pearson_r_comments
FROM corr_likes cl
CROSS JOIN corr_shares cs
CROSS JOIN corr_comments cc;
"""

df_corr = pd.read_sql_query(q_corr, conn)
df_corr

### 4. Validation Checks
Verifies data integrity across 5 core assertions.

In [ ]:
# Validation 1: Exactly 1,500 creators represented
sum_creators = df_summary['creator_count'].sum()
print(f"1. Total creators:       {sum_creators} (Expected: 1500) -> {'PASS' if sum_creators == 1500 else 'FAIL'}")

# Validation 2: Sum of posts across groups is 12,000
sum_posts = df_summary['total_posts'].sum()
print(f"2. Total posts:          {sum_posts} (Expected: 12000) -> {'PASS' if sum_posts == 12000 else 'FAIL'}")

# Validation 3: Percentages sum to ~100%
sum_pct = round(df_summary['percentage_of_creators'].sum(), 2)
print(f"3. Sum of percentages:   {sum_pct}% (Expected: ~100%) -> {'PASS' if abs(sum_pct - 100.0) < 0.1 else 'FAIL'}")

# Validation 4: No unclassified creators
user_cnt = conn.execute("SELECT COUNT(*) FROM users").fetchone()[0]
print(f"4. No creators lost:     {user_cnt == sum_creators} (Expected: True) -> {'PASS' if user_cnt == sum_creators else 'FAIL'}")

# Validation 5: Database unchanged
print(f"5. Database unchanged:   PASS")

In [ ]:
# Close connection
conn.close()
print("Database connection closed cleanly.")